In [12]:
import numpy as np
import pandas as pd
import seaborn as sns
import math
import os
import pathlib
import trackpy as tp
import napari
import imageio

from joblib import Parallel, delayed
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage import util,  restoration
from skimage.filters import threshold_otsu, threshold_yen, gaussian
from skimage.segmentation import clear_border, expand_labels
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, closing,  disk, dilation, erosion
from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi
from tifffile import imread
import tifffile
import skimage
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import pathlib
from PIL import Image
from cellpose import plot
from cellpose import models, utils, io

from skimage import color
from skimage import io
import napari
from matplotlib.ticker import MaxNLocator
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile as tiff
import imagej
from imagej import Mode
import scyjava as sj
import contextlib
import joblib
from tqdm import tqdm
import os, sys, glob, platform
from pathlib import Path
import tifffile as tiff
import numpy as np
import pandas as pd

from ultrack import MainConfig, Tracker
# import mip  # python-mip

In [13]:
def cellpose_live_segmentation(stack, to_plot=False, to_save=False, 
                               diameter=None, flow_threshold=0.4, 
                               cellprob_threshold=0.0, min_size=15):
    """
    Segment cells slice-by-slice using Cellpose 3.0 (cyto3 model) for brightfield stacks.
    Optimized settings to match micro-sam quality with GPU acceleration (RTX 3080).
    
    Parameters:
    -----------
    stack : str or ndarray
        Path to image file or numpy array of shape (T, H, W) or (H, W)
    to_plot : bool
        Whether to plot results (optional, not implemented)
    to_save : bool
        Whether to save results (optional, not implemented)
    diameter : float or None
        Expected cell diameter in pixels. None = auto-detect from first frame
    flow_threshold : float (default 0.4)
        Flow error threshold (0-1). Lower = more cells, higher = fewer cells
        Recommend 0.4 for brightfield (default), 0.6-0.8 for cleaner images
    cellprob_threshold : float (default 0.0)
        Cell probability threshold (-6 to 6). Higher = stricter segmentation
        Recommend 0.0 (default) for brightfield, 1.0-2.0 for noisy images
    min_size : int (default 15)
        Minimum cell size in pixels (removes small debris)
    
    Returns:
    --------
    masks_stack : ndarray
        Segmentation masks with same shape as input stack
    
    Notes:
    ------
    - Uses Cellpose 3.0's cyto3 model (transformer-based, comparable to SAM)
    - Optimized for brightfield cell segmentation
    - Auto-detects diameter from first frame if not specified
    - GPU-accelerated on CUDA-compatible devices
    """
    # Load image if path is provided
    if isinstance(stack, (str, pathlib.Path)):
        stack = io.imread(stack)
    
    # Handle single image (2D)
    if stack.ndim == 2:
        stack = stack[np.newaxis, :, :]
    
    # Initialize Cellpose 3.0 cyto3 model (transformer architecture, similar to micro-sam)
    # This is the latest and most accurate general cell segmentation model
    model = models.CellposeModel(gpu=True, model_type='cyto3')
    
    # Auto-detect diameter from first frame if not provided
    if diameter is None:
        print("Auto-detecting cell diameter from first frame...")
        masks_first, _, _ = model.eval(
            stack[0, :, :], 
            diameter=None,
            channels=[0, 0],
            flow_threshold=flow_threshold,
            cellprob_threshold=cellprob_threshold
        )
        # Estimate diameter from detected cell sizes
        from scipy import ndimage
        if masks_first.max() > 0:
            cell_areas = [np.sum(masks_first == i) for i in range(1, min(masks_first.max() + 1, 50))]
            median_area = np.median(cell_areas) if cell_areas else 700
            diameter = 2 * np.sqrt(median_area / np.pi)  # Convert area to diameter
        else:
            diameter = 30.0  # Default fallback
        print(f"  Estimated diameter: {diameter:.1f} pixels")
    
    # Preallocate output array
    masks_stack = np.zeros(stack.shape, dtype=np.uint16)
    
    # Segment each slice with progress bar
    print(f"Segmenting {stack.shape[0]} frames with Cellpose cyto3 (GPU)...")
    for z in tqdm(range(stack.shape[0]), desc="Cellpose segmentation"):
        # channels=[0,0] = grayscale brightfield
        # resample=True improves accuracy for cells of varying sizes
        masks, flows, styles = model.eval(
            stack[z, :, :], 
            diameter=diameter,
            channels=[0, 0],
            flow_threshold=flow_threshold,
            cellprob_threshold=cellprob_threshold,
            min_size=min_size,
            resample=True  # Better for varying cell sizes
        )
        masks_stack[z, :, :] = masks.astype(np.uint16)
    
    # Summary statistics
    cell_counts = [masks_stack[i].max() for i in range(min(5, stack.shape[0]))]
    total_cells = sum(frame.max() for frame in masks_stack)
    print(f"\n✓ Segmentation complete!")
    print(f"  Diameter used: {diameter:.1f} pixels")
    print(f"  First 5 frames: {cell_counts} cells")
    print(f"  Total cells detected: {total_cells}")
    
    # Return single 2D mask if input was 2D
    if masks_stack.shape[0] == 1:
        return masks_stack[0]
    
    return masks_stack

In [14]:
output_directory = Path(r"e:\Marcus\cellpose_outputs")
output_directory.mkdir(parents=True, exist_ok=True)
print(f"Output directory ready: {output_directory.resolve()}")

Output directory ready: E:\Marcus\cellpose_outputs


In [15]:
image = imread(r"e:\Marcus\combined_stack.tif")
brightfield_image = image[:, -1, :, :]
red_fluorescence_image = image[:, 0, :, :]
green_fluorescence_image = image[:, 1, :, :]

masks_stack = cellpose_live_segmentation(brightfield_image, to_plot=True, to_save=True)
masks_path = output_directory / "masks_stack.tiff"
tiff.imwrite(str(masks_path), masks_stack.astype(np.uint16))
print(f"Saved masks stack to: {masks_path.resolve()}")

Auto-detecting cell diameter from first frame...
  Estimated diameter: 48.1 pixels
Segmenting 25 frames with Cellpose cyto3 (GPU)...


Cellpose segmentation: 100%|██████████| 25/25 [00:27<00:00,  1.10s/it]



✓ Segmentation complete!
  Diameter used: 48.1 pixels
  First 5 frames: [np.uint16(199), np.uint16(211), np.uint16(209), np.uint16(211), np.uint16(208)] cells
  Total cells detected: 5360
Saved masks stack to: E:\Marcus\cellpose_outputs\masks_stack.tiff


In [19]:
# 3) Initialize Fiji headless through PyImageJ

# Ensure a JVM is discoverable for PyImageJ/JPype
if "JAVA_HOME" not in os.environ or not os.environ.get("JAVA_HOME"):
    candidate_homes = []
    try:
        import jdk4py
        candidate_homes.append(str(jdk4py.JAVA_HOME))
    except Exception:
        pass
    candidate_homes.append(str(Path(sys.prefix) / "Library"))

    configured = False
    for home in candidate_homes:
        jvm_dll = Path(home) / "bin" / "server" / "jvm.dll"
        if jvm_dll.exists():
            os.environ["JAVA_HOME"] = str(home)
            os.environ["PATH"] = str(Path(home) / "bin") + os.pathsep + os.environ.get("PATH", "")
            print(f"JAVA_HOME set to: {os.environ['JAVA_HOME']}")
            configured = True
            break

    if not configured:
        print("Could not auto-configure JAVA_HOME; no valid jvm.dll found in candidates.")

print("Initializing PyImageJ headless (first run may download Fiji components)...")
ij = imagej.init("sc.fiji:fiji", mode=Mode.HEADLESS, add_legacy=True)

# Requested TrackMate configuration
TARGET_CHANNEL = 1
SIMPLIFY_CONTOURS = False
INITIAL_SEARCH_RADIUS = 30.0
SEARCH_RADIUS = 150.0
MAX_FRAME_GAP = 3
ALLOW_TRACK_SPLITTING = False
ALLOW_TRACK_MERGING = False
DEBUG_TRACKMATE = True

# Unified output directory for all generated files
TRACKMATE_WORKDIR = Path(output_directory)
TRACKMATE_WORKDIR.mkdir(parents=True, exist_ok=True)

# Resolve masks path from unified output directory
masks_path = TRACKMATE_WORKDIR / "masks_stack.tiff"
if not masks_path.exists() and "masks_stack" in globals():
    tiff.imwrite(str(masks_path), np.asarray(masks_stack).astype(np.uint16))
    print(f"Re-saved masks stack to: {masks_path.resolve()}")

# Java imports
IJ = sj.jimport("ij.IJ")
HashMap = sj.jimport("java.util.HashMap")
ArrayList = sj.jimport("java.util.ArrayList")
Integer = sj.jimport("java.lang.Integer")
Double = sj.jimport("java.lang.Double")

Model = sj.jimport("fiji.plugin.trackmate.Model")
Settings = sj.jimport("fiji.plugin.trackmate.Settings")
TrackMate = sj.jimport("fiji.plugin.trackmate.TrackMate")
Logger = sj.jimport("fiji.plugin.trackmate.Logger")

LabelImageDetectorFactory = sj.jimport("fiji.plugin.trackmate.detection.LabelImageDetectorFactory")
AdvancedKalmanTrackerFactory = sj.jimport("fiji.plugin.trackmate.tracking.kalman.AdvancedKalmanTrackerFactory")

# 4) Run TrackMate with label-image detector + advanced Kalman tracker
imp = IJ.openImage(str(masks_path))
if imp is None:
    raise RuntimeError(f"Could not open image: {masks_path}")

# Force interpretation as T,Y,X (not C,Y,X or Z,Y,X): C=1, Z=1, T=n_timepoints
n_timepoints = int(imp.getNFrames())
if n_timepoints <= 1:
    n_timepoints = int(imp.getStackSize())
imp.setDimensions(1, 1, n_timepoints)
imp.setOpenAsHyperStack(True)

model = Model()
model.setLogger(Logger.IJ_LOGGER)

settings = Settings(imp)
settings.detectorFactory = LabelImageDetectorFactory()
detector_settings = HashMap()
detector_settings.put("TARGET_CHANNEL", Integer.valueOf(int(TARGET_CHANNEL)))
detector_settings.put("SIMPLIFY_CONTOURS", bool(SIMPLIFY_CONTOURS))
settings.detectorSettings = detector_settings

settings.trackerFactory = AdvancedKalmanTrackerFactory()
tracker_settings = HashMap(settings.trackerFactory.getDefaultSettings())
tracker_settings.put("KALMAN_SEARCH_RADIUS", Double.valueOf(float(SEARCH_RADIUS)))
tracker_settings.put("LINKING_MAX_DISTANCE", Double.valueOf(float(INITIAL_SEARCH_RADIUS)))
tracker_settings.put("MAX_FRAME_GAP", Integer.valueOf(int(MAX_FRAME_GAP)))
tracker_settings.put("ALLOW_TRACK_SPLITTING", ALLOW_TRACK_SPLITTING)
tracker_settings.put("ALLOW_TRACK_MERGING", ALLOW_TRACK_MERGING)

# Feature penalties: area, x, y all = 1.0 ("xy" represented via X and Y penalties)
feature_penalties = HashMap()
feature_penalties.put("POSITION_X", Double.valueOf(1.0))
feature_penalties.put("POSITION_Y", Double.valueOf(1.0))
feature_penalties.put("AREA", Double.valueOf(1.0))
tracker_settings.put("LINKING_FEATURE_PENALTIES", feature_penalties)
tracker_settings.put("GAP_CLOSING_FEATURE_PENALTIES", feature_penalties)

settings.trackerSettings = tracker_settings
settings.addAllAnalyzers()

if DEBUG_TRACKMATE:
    print("=== TrackMate settings debug ===")
    print(f"Detector factory: {settings.detectorFactory}")
    print(f"Detector settings: {dict(detector_settings)}")
    print("Tracker settings keys:", list(tracker_settings.keySet()))
    print(
        "Tracker params:",
        f"LINKING_MAX_DISTANCE={tracker_settings.get('LINKING_MAX_DISTANCE')}, "
        f"KALMAN_SEARCH_RADIUS={tracker_settings.get('KALMAN_SEARCH_RADIUS')}, "
        f"MAX_FRAME_GAP={tracker_settings.get('MAX_FRAME_GAP')}, "
        f"ALLOW_TRACK_SPLITTING={tracker_settings.get('ALLOW_TRACK_SPLITTING')}, "
        f"ALLOW_TRACK_MERGING={tracker_settings.get('ALLOW_TRACK_MERGING')}"
    )

trackmate = TrackMate(model, settings)
if not trackmate.checkInput():
    raise RuntimeError(f"TrackMate input error: {trackmate.getErrorMessage()}")
if not trackmate.process():
    raise RuntimeError(f"TrackMate process error: {trackmate.getErrorMessage()}")

# Debug: detected spots vs linked tracks
spots = model.getSpots()
n_spots = int(spots.getNSpots(True))
tm = model.getTrackModel()
track_ids = list(tm.trackIDs(True))
n_tracks = len(track_ids)
if DEBUG_TRACKMATE:
    print("=== Output diagnostics ===")
    print(f"Detected spots (all frames): {n_spots}")
    print(f"Linked tracks: {n_tracks}")

    # Spot count per frame
    try:
        spot_rows = []
        for frame in range(int(n_timepoints)):
            n_frame_spots = int(spots.getNSpots(frame, True))
            spot_rows.append({"t": frame, "n_detected_spots": n_frame_spots})
        spots_per_frame_df = pd.DataFrame(spot_rows)
        print("Detected spots per frame (first 10):")
        display(spots_per_frame_df.head(10))
    except Exception as e:
        print(f"Could not compute spots-per-frame breakdown: {e}")

# 5) Export tracks to DataFrame + CSV
rows = []
for track_id in track_ids:
    track_spots = list(tm.trackSpots(track_id))
    track_spots.sort(key=lambda s: float(s.getFeature("FRAME")))

    for spot in track_spots:
        t = int(float(spot.getFeature("FRAME")))
        x = float(spot.getFeature("POSITION_X"))
        y = float(spot.getFeature("POSITION_Y"))
        q = float(spot.getFeature("QUALITY"))
        rows.append({"track_id": int(track_id), "t": t, "y": y, "x": x, "quality": q})

trackmate_tracks_df = pd.DataFrame(rows, columns=["track_id", "t", "y", "x", "quality"])
tracks_csv = TRACKMATE_WORKDIR / "trackmate_tracks.csv"
trackmate_tracks_df.to_csv(tracks_csv, index=False)

# Save linked label image once (for fast Napari reload later)
linked_labels_path = TRACKMATE_WORKDIR / "linked_labels_trackmate.tiff"
masks_stack_local = tiff.imread(str(masks_path))
linked_labels = np.zeros_like(masks_stack_local, dtype=np.uint32)
linked_track_points = trackmate_tracks_df[["track_id", "t", "y", "x"]].to_numpy()
n_t, n_y, n_x = masks_stack_local.shape

for track_id, t, y, x in linked_track_points:
    ti = int(t)
    yi = int(np.clip(round(float(y)), 0, n_y - 1))
    xi = int(np.clip(round(float(x)), 0, n_x - 1))
    if ti < 0 or ti >= n_t:
        continue

    label_id = int(masks_stack_local[ti, yi, xi])
    if label_id <= 0:
        continue

    track_label_value = int(track_id) + 1
    linked_labels[ti, masks_stack_local[ti] == label_id] = track_label_value

tiff.imwrite(str(linked_labels_path), linked_labels)

if DEBUG_TRACKMATE and len(trackmate_tracks_df) > 0:
    track_len = trackmate_tracks_df.groupby("track_id").size()
    print("Track length summary:")
    print(track_len.describe())

print(f"Saved TrackMate CSV: {tracks_csv.resolve()}")
print(f"Saved linked labels TIFF: {linked_labels_path.resolve()}")
print(f"Loaded TrackMate tracks: {len(trackmate_tracks_df)} rows")
display(trackmate_tracks_df.head(10))

Initializing PyImageJ headless (first run may download Fiji components)...
=== TrackMate settings debug ===
Detector factory: fiji.plugin.trackmate.detection.LabelImageDetectorFactory@37344d31
Detector settings: {'TARGET_CHANNEL': 1, 'SIMPLIFY_CONTOURS': False}
Tracker settings keys: ['MAX_FRAME_GAP', 'ALTERNATIVE_LINKING_COST_FACTOR', 'KALMAN_SEARCH_RADIUS', 'LINKING_FEATURE_PENALTIES', 'LINKING_MAX_DISTANCE', 'GAP_CLOSING_MAX_DISTANCE', 'MERGING_FEATURE_PENALTIES', 'SPLITTING_MAX_DISTANCE', 'BLOCKING_VALUE', 'ALLOW_GAP_CLOSING', 'ALLOW_TRACK_SPLITTING', 'ALLOW_TRACK_MERGING', 'MERGING_MAX_DISTANCE', 'SPLITTING_FEATURE_PENALTIES', 'CUTOFF_PERCENTILE', 'GAP_CLOSING_FEATURE_PENALTIES']
Tracker params: LINKING_MAX_DISTANCE=30.0, KALMAN_SEARCH_RADIUS=150.0, MAX_FRAME_GAP=3, ALLOW_TRACK_SPLITTING=False, ALLOW_TRACK_MERGING=False
=== Output diagnostics ===
Detected spots (all frames): 5391
Linked tracks: 282
Detected spots per frame (first 10):


,t,n_detected_spots
0,0,199
1,1,211
2,2,209
3,3,212
4,4,208
5,5,223
6,6,218
7,7,233
8,8,207
9,9,228


Track length summary:
count    282.000000
mean      16.436170
std        7.049735
min        2.000000
25%       10.250000
50%       18.000000
75%       23.000000
max       25.000000
dtype: float64
Saved TrackMate CSV: E:\Marcus\cellpose_outputs\trackmate_tracks.csv
Saved linked labels TIFF: E:\Marcus\cellpose_outputs\linked_labels_trackmate.tiff
Loaded TrackMate tracks: 4635 rows


,track_id,t,y,x,quality
0,0,0,193.770690,1400.272414,580.0
1,0,1,184.373723,1400.553285,685.0
2,0,2,175.250840,1398.097424,893.0
3,0,3,168.000840,1394.768908,1190.0
4,0,4,162.094692,1397.784792,697.0
5,0,5,158.318052,1401.303725,349.0
6,0,6,143.038462,1400.623482,494.0
7,0,7,156.969369,1398.684685,555.0
8,0,8,191.548443,1401.536332,578.0
9,0,9,173.998138,1400.309125,537.0


In [21]:
# Optional: quick Napari overlay of TrackMate tracks + linked labels
if "trackmate_tracks_df" not in globals():
    raise ValueError("Run Cell 5 (TrackMate headless) first.")

if "masks_stack" not in globals():
    masks_path = Path(output_directory) / "masks_stack.tiff"
    if not masks_path.exists():
        raise ValueError(f"Could not find masks stack at: {masks_path}")
    masks_stack = tiff.imread(str(masks_path))

linked_labels_path = Path(output_directory) / "linked_labels_trackmate.tiff"
if not linked_labels_path.exists():
    raise ValueError(
        f"Linked labels not found at: {linked_labels_path}. "
        "Run Cell 5 first to generate it."
    )
linked_labels = tiff.imread(str(linked_labels_path))

viewer = napari.Viewer()
viewer.add_image(image[:, -1, :, :], name="Raw Image", colormap="gray")

required_cols = ["track_id", "t", "y", "x"]
missing_cols = [c for c in required_cols if c not in trackmate_tracks_df.columns]
if missing_cols:
    raise ValueError(f"`trackmate_tracks_df` missing columns: {missing_cols}")

if len(trackmate_tracks_df) == 0:
    print("No tracks to display (trackmate_tracks_df is empty).")
else:
    tracks_for_napari = trackmate_tracks_df[required_cols].to_numpy()
    viewer.add_tracks(tracks_for_napari, name="TrackMate tracks")
    viewer.add_labels(linked_labels, name="Linked labels", opacity=0.35)
    print(f"Displayed {len(tracks_for_napari)} track points in napari.")
    print(f"Loaded linked labels from: {linked_labels_path}")

Displayed 4635 track points in napari.
Loaded linked labels from: e:\Marcus\cellpose_outputs\linked_labels_trackmate.tiff


<Labels layer 'Cellpose Segmentation' at 0x25463b8ba90>